# SafeSite AI — YOLOv8 PPE Training Notebook

**Platform:** Google Colab / Kaggle (GPU required)

**Dataset:** Roboflow Construction Site Safety (v28)

**Steps:**
1. Set runtime to **T4 GPU** → Runtime → Change runtime type → T4 GPU
2. Fill in your Roboflow API key in Cell 2
3. Run all cells top to bottom (Runtime → Run all)
4. Download `best.pt` from Cell 5
5. Place it in `backend/weights/best.pt` in your local SafeSite AI project

In [ ]:
# Cell 1 — Install dependencies
!pip install ultralytics roboflow -q

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected. Training will be very slow on CPU.')
    print('Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Cell 2 — Download dataset via Roboflow
# Get your API key from: https://app.roboflow.com → Settings → API Keys
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"   # ← replace this

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
dataset = project.version(28).download("yolov8")

DATASET_YAML = dataset.location + "/data.yaml"
print(f"\nDataset downloaded to: {dataset.location}")
print(f"Dataset YAML: {DATASET_YAML}")

In [ ]:
# Cell 3 — Train YOLOv8n
# yolov8n = nano (fastest, good enough for PPE)
# Change to yolov8s.pt for better accuracy (+5 min training)
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=DATASET_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,          # GPU
    workers=2,
    project="runs/train",
    name="ppe_detector",
    exist_ok=True,
    patience=15,
    save=True,
    plots=True,
    # Augmentation tuned for construction sites
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    mixup=0.1,
)

print("\nTraining complete!")

In [ ]:
# Cell 4 — Evaluate the trained model
import json
from pathlib import Path

BEST_PT = "runs/train/ppe_detector/weights/best.pt"

eval_model = YOLO(BEST_PT)
metrics = eval_model.val(data=DATASET_YAML, device=0, verbose=False)

rd = metrics.results_dict
map50   = rd.get('metrics/mAP50(B)', 0)
map5095 = rd.get('metrics/mAP50-95(B)', 0)
prec    = rd.get('metrics/precision(B)', 0)
recall  = rd.get('metrics/recall(B)', 0)

print("=" * 45)
print("  Validation Metrics")
print("=" * 45)
print(f"  mAP50    : {map50:.4f}  {'✓ Good' if map50 >= 0.7 else '~ Acceptable' if map50 >= 0.5 else '✗ Low'}")
print(f"  mAP50-95 : {map5095:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall   : {recall:.4f}")

# Per-class
if hasattr(metrics, 'ap_class_index') and metrics.ap_class_index is not None:
    print("\n  Per-class mAP50:")
    for idx, ap in zip(metrics.ap_class_index, metrics.box.ap50):
        cls = eval_model.names.get(int(idx), str(idx))
        flag = " ← violation class" if cls in ("NO-Hardhat", "NO-Safety Vest") else ""
        print(f"    {cls:<22} {float(ap):.4f}{flag}")
print("=" * 45)

if map50 < 0.50:
    print("\nTip: Low mAP. Try training yolov8s.pt for 100 epochs.")

In [ ]:
# Cell 5 — Download best.pt
# This downloads best.pt to your local machine.
# Place it at: SafeSite-AI/backend/weights/best.pt

from google.colab import files
import shutil

BEST_PT = "runs/train/ppe_detector/weights/best.pt"

# Also export to ONNX for faster CPU inference (optional)
# Uncomment the next 2 lines if you want ONNX:
# onnx_model = YOLO(BEST_PT)
# onnx_model.export(format="onnx", imgsz=640, simplify=True)

print(f"Downloading {BEST_PT} ...")
files.download(BEST_PT)

print("\n" + "=" * 50)
print("  Next steps:")
print("  1. Save the downloaded best.pt")
print("  2. Place it in your project:")
print("     SafeSite-AI/backend/weights/best.pt")
print("  3. Test locally:")
print("     python training/inference_video.py --source video.mp4")
print("  4. Evaluate:")
print("     python training/evaluate_model.py")
print("=" * 50)

In [ ]:
# Cell 6 — Kaggle users: use this download instead of Cell 5
# (Kaggle does not have google.colab.files)
#
# import shutil
# shutil.copy("runs/train/ppe_detector/weights/best.pt", "/kaggle/working/best.pt")
# print("best.pt is now in /kaggle/working/ — download from the Output tab")